## Simulated survey:
1. pool of R respondents each with preference vector beta_r
2. run batches of T tasks with R respondents and m expected contrasts/task
3. before first batch: compute + normalize sampling weights to assign baseline p_flip
4. afer each batch: pool comparisons + apply move-on rule for significant features to get updated p_flip

In [265]:
import pandas as pd
import numpy as np
from numpy.typing import NDArray

In [255]:
import sys

In [243]:
import statsmodels.api as sm

In [251]:
from scipy.stats import norm

In [138]:
rand = np.random.default_rng(seed=42)

In [ ]:
# global vars for
# - number of features/attributes
# - number of tasks/batch
# - number of respondents/batch
# - (sigma)^2: heterogeneity in respondent preferences
# - power-law concentration
# - expected number of flipped features/task

In [209]:
P = 12
T = 10
R = 150
RES_VAR = 0.10
gamma = 1.2
m = 2

### Pool of simulated respondents

In [ ]:
# get B_true baseline with index-based power-law decay: 

In [174]:
k = np.arange(1, P + 1)

In [175]:
C = 1 / np.sum(k ** (-gamma))

In [176]:
B_true = C * k ** (-gamma)

In [177]:
var_matrix = RES_VAR * np.eye(P)

In [178]:
# add heterogeneity and generate full pool of respondents 

In [179]:
B = rand.multivariate_normal(mean=B_true, cov=var_matrix, size=R)
B.shape

(150, 12)

### "Images"

In [170]:
features = pd.read_csv("images/data-center-protests/simple_features.csv")

In [194]:
feature_cols = features.columns[1:]
features["id"] = features[feature_cols].astype(str).agg("".join, axis=1)

In [195]:
features

,id,setting_is_rural,race_white,race_black,race_hispanic,race_asian,age_old,age_young,sign_amazon,sign_google,sign_microsoft,sign_no_company,focal_single
0,000000000000,0,0,0,0,0,0,0,0,0,0,0,0
1,000000000001,0,0,0,0,0,0,0,0,0,0,0,1
2,000000000010,0,0,0,0,0,0,0,0,0,0,1,0
3,000000000011,0,0,0,0,0,0,0,0,0,0,1,1
4,000000000100,0,0,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4091,111111111011,1,1,1,1,1,1,1,1,1,0,1,1
4092,111111111100,1,1,1,1,1,1,1,1,1,1,0,0
4093,111111111101,1,1,1,1,1,1,1,1,1,1,0,1
4094,111111111110,1,1,1,1,1,1,1,1,1,1,1,0


### Running batches (batch-adaptive, significance-gated move-on strategy)

In [180]:
def flip_probs(preference):
    if (np.sum(np.abs(preference)) > 0):
        weights = np.abs(preference) / np.sum(np.abs(preference))
    else:
        weights = np.full(P, 1/P)
    p_flip = m * weights
    return p_flip

In [181]:
# baseline example:
p_flip = flip_probs(B_true)
rand.binomial(1, p_flip, size=P)

array([1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [258]:
def move_on(S_b, preference):
    raw_weights = np.abs(preference).copy()
    raw_weights[list(S_b)] = sys.float_info.epsilon
    weights = raw_weights / raw_weights.sum()
    p_flip = m * weights
    return p_flip

In [262]:
def run_batches(init_preferences, B_population, data, num_B):
    comparisons = []
    choices = []

    # initial contrast probablities
    p_flip = flip_probs(init_preferences)
    
    for b in range(num_B):
        # generate T tasks for each respondent with current contrast prob
        for r in B_population:
            for t in range(T):
                # for current csv setup with id column describing feature vector (i.e. '10001000101000101')
                # note no mutually exclusive features in test set (with full 2^12 combination) for simplicity 
                # realistically would need to handle when users upload data without all combinations
                img1 = np.fromiter(rand.choice(data["id"].values), dtype=int)
                flips = rand.binomial(1, p_flip, size=P)
                img2 = img1 ^ flips
    
                # r "chooses" img1 or img2
                d = img1 - img2
                utility_diff = np.dot(d, r)
                p_img1 = 1 / (1 + np.exp(-utility_diff))
                choice = rand.binomial(1, p_img1)
    
                # keep track of comparisons
                comparisons.append(d)
                choices.append(choice)
                
        # get pooled population estimate
        model = sm.Logit(choices, comparisons)
        result = model.fit()
        B_pooled = result.params

        # apply move-on rule and update
        alpha = 0.05
        critical_value = norm.ppf(1 - alpha / 2)
        SE = result.bse
        S_b = set(np.where(np.abs(B_pooled/SE) > critical_value)[0])
        p_flip = move_on(S_b, B_pooled)

        # print end of batch pooled comparisons, choices, preference:
        print("finished batch:", b)
        print("B_pooled:", B_pooled)
        print("S_b:", S_b)
        print("p_flip:", p_flip)
        print("sum(p_flip):", p_flip.sum())

In [223]:
x = np.fromiter(rand.choice(features["id"].values), dtype=int)
x

array([1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0])

In [224]:
y = rand.binomial(1, p_flip, size=P)
y

array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [225]:
x ^ y

array([0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0])

### Simulated Run:

In [260]:
run_batches(B_true, B, features, 20)

Optimization terminated successfully.
         Current function value: 0.673213
         Iterations 4
finished batch: 0
B_pooled: [ 0.35585908  0.34961986  0.08505969  0.19619474  0.01455383 -0.15529081
 -0.29158341  0.04443727  0.08579064  0.32778397  0.43376837 -0.00507454]
S_b: {np.int64(0), np.int64(1)}
p_flip: [2.70862529e-16 2.70862529e-16 1.03760600e-01 2.39329402e-01
 1.77535788e-02 1.89432484e-01 3.55689884e-01 5.42070885e-02
 1.04652265e-01 3.99849364e-01 5.29135121e-01 6.19021260e-03]
sum(p_flip): 2.0
Optimization terminated successfully.
         Current function value: 0.683888
         Iterations 4
finished batch: 1
B_pooled: [ 0.34909063  0.35187018  0.14015009  0.03549514  0.02899042  0.07970573
 -0.03992103  0.06239069 -0.02357951  0.08385383  0.08950473  0.04703691]
S_b: {np.int64(0), np.int64(1)}
p_flip: [7.04201446e-16 7.04201446e-16 4.44477798e-01 1.12570768e-01
 9.19414279e-02 2.52782055e-01 1.26607206e-01 1.97868401e-01
 7.47810251e-02 2.65937508e-01 2.83858994e-

ValueError: p < 0, p > 1 or p contains NaNs